# 03 — Our Pipeline, Annotated

**Estimated reading time: ~35 minutes** (the code cells each run in seconds, on CPU).

This notebook walks through the *actual* source files of LongFlow's P1 pipeline, in plain
language, so that afterwards you can open any of them and recognize every trick. Everything
quoted below is real code from this repo — not simplified pseudo-code. The files covered:

| File | What it does |
|---|---|
| `src/cache/capture.py` | Steals perfectly-aligned training pairs out of a running VibeVoice |
| `src/cache/alignment.py` | The 10-line guard that refuses to let April 7 happen again |
| `src/flow_head/trainer.py` | Trains the 15M-param flow head on the cached pairs |
| `src/eval/metrics.py` | Measures whether the result is actually any good |

Along the way, every Python idiom that shows up in those files gets a short
**Python syntax note** box.

**How to run:** start Jupyter from the repo root (e.g. `uv run jupyter lab`), open this
notebook, and run cells top to bottom — one early cell adds the repo root to `sys.path`
so `import src...` works. No GPU, no model downloads, no `vibevoice` import anywhere.


## 1. The caching idea in one page

To train a new, faster speech head we need training pairs of **(thought, sound)**:

- the **thought**: the language model's hidden state right before it emits one frame of
  speech — a vector of 1,536 numbers describing "what I intend to say next";
- the **sound**: the acoustic latent the original diffusion head actually produced for that
  frame — 64 numbers that the frozen σ-VAE decoder later turns into ~133 ms of audio.

Getting these two things *and keeping them lined up frame-for-frame* sounds like it should
require surgery deep inside the model. It doesn't, because of a lucky architectural fact,
documented at the top of `src/cache/capture.py`:

> Hook point: VibeVoice's `sample_speech_tokens(condition, neg_condition, cfg_scale)`
> is called **exactly once per speech frame** during full inference (feedback loop
> included — hard constraint 2). Its `condition` argument IS the LM hidden state
> the diffusion head conditions on (`[B, d_model]`); its return IS the sampled
> acoustic latent (`[B, d_latent]`, in the head's scaled space — the same space the
> flow head must learn to generate in). Wrapping this one callable yields exactly
> the training pairs, **pre-aligned by construction**, with no layer-hook reassembly.

Every frame of every utterance funnels through that one function. So instead of attaching
hooks to internal layers and trying to re-associate tensors afterwards, we simply **wrap
that one function**: our wrapper calls the original, writes down its input and its output,
and hands the output back untouched. The model never notices. One call = one frame, so the
input–output pair *cannot* be misaligned — alignment is guaranteed by construction, not by
bookkeeping.

### Monkeypatching, in one breath

Swapping out a method on a live object at runtime is called **monkeypatching**. In Python,
methods are just attributes, and attributes can be reassigned — so you can grab the
original (`orig = getattr(model, "sample_speech_tokens")`), install your own function in
its place (`setattr(model, ..., wrapped)`), and put things back when you're done. That
last part — *always* putting things back — is why `SampleCapture` is written as a context
manager.

> **Python syntax note — `with` / context managers.** `with SampleCapture(model) as cap:`
> creates an object whose `__enter__` method runs when the block starts and whose
> `__exit__` method runs when the block ends — **even if an exception blows up in the
> middle**. It's Python's "no matter what happens, clean up" construct, ideal for
> install-wrapper / always-uninstall-wrapper.

There is one genuinely sneaky detail, and we hit it: **the instance-attribute shadow**.
When you write `model.method = wrapped`, you usually don't overwrite the method defined on
the *class* — you create a new attribute on the *instance* that **shadows** it. On exit,
naively assigning the original back would leave a permanent instance attribute lying
around. The correct cleanup depends on where the original lived, which is why
`SampleCapture.__enter__` starts with this check:

```python
# If the attribute lives on the instance we must restore it on exit;
# if it lives on the class we must delete our instance-level shadow.
self._was_instance_attr = self.method_name in vars(self.model)
```

`vars(obj)` returns the instance's own attribute dictionary — the class's methods are
*not* in it. The toy below reproduces the whole dance, gotcha included.


In [ ]:
# Toy monkeypatch: wrap a method on a dummy object, capture, restore.
# Pure Python -- no imports needed.

class Synth:                       # stand-in for the VibeVoice model
    def sample(self, condition):   # stand-in for sample_speech_tokens
        return condition * 2       # "generate a latent from a hidden state"

synth = Synth()
captured = []
original = synth.sample            # bound method, looked up on the CLASS

def wrapped(condition):
    out = original(condition)                # 1. call the real thing
    captured.append((condition, out))        # 2. write down input and output
    return out                               # 3. hand the result back untouched

print("vars(synth) before patch:", vars(synth))   # {} -- 'sample' lives on the class
synth.sample = wrapped             # monkeypatch: creates an INSTANCE attribute...
print("vars(synth) after patch: ", vars(synth))   # ...that SHADOWS the class method

print("calls still work:", synth.sample(3), synth.sample(10))
print("captured pairs:  ", captured)

# --- restoration: the gotcha we hit ---
# WRONG:  synth.sample = original   -> works, but leaves a permanent instance
#         attribute shadowing the class method forever (vars() stays polluted,
#         and a second capture would misdetect where the method lives).
# RIGHT:  delete the shadow; attribute lookup falls back to the class.
del synth.sample
print("vars(synth) after cleanup:", vars(synth))  # {} again -- pristine
print("still works via the class:", synth.sample(3))


### The real thing: `SampleCapture`

`src/cache/capture.py` does exactly what the toy did, plus three production details.
Here is the heart of it:

```python
def __enter__(self):
    self._was_instance_attr = self.method_name in vars(self.model)
    self._orig = getattr(self.model, self.method_name)
    capture = self

    def wrapped(condition, *args, **kwargs):
        out = capture._orig(condition, *args, **kwargs)
        capture.conditions.append(condition.detach().to("cpu", torch.float16))
        capture.latents.append(out.detach().to("cpu", torch.float16))
        return out

    setattr(self.model, self.method_name, wrapped)
    return self

def __exit__(self, *exc):
    if self._was_instance_attr:
        setattr(self.model, self.method_name, self._orig)
    else:
        delattr(self.model, self.method_name)  # restore class-method lookup
    self._orig = None
    return False
```

The three production details:

1. **`.detach().to("cpu", torch.float16)`** — drop autograd bookkeeping (`detach`), move
   the tensor off the GPU, and store it at half precision. The GPU stays free for
   generation; the cache costs half the memory (more on fp16 in section 3).
2. **Duck typing** — `SampleCapture` never imports VibeVoice. It works on *any* object
   that has a method with the right name, which is why a 5-line stub model can unit-test
   it (and why this notebook can demo it without downloading anything).
3. Captured frames are bundled into an `UtteranceCache`:

```python
@dataclass
class UtteranceCache:
    utt_id: str
    text: str
    hidden: torch.Tensor  # [T, d_model] fp16
    latent: torch.Tensor  # [T, d_latent] fp16
    meta: dict = field(default_factory=dict)
```

> **Python syntax note — `@dataclass`.** A decorator that auto-writes the boring parts of
> a class: `__init__`, `__repr__`, equality. You list the fields with type hints; Python
> generates the constructor. `field(default_factory=dict)` means "each instance gets its
> **own** fresh empty dict" — a plain `meta: dict = {}` default would dangerously share
> one dict across all instances.

> **Python syntax note — `@property`.** `SampleCapture` defines
> `num_frames` with `@property`, so you write `cap.num_frames` (no parentheses) and a
> method runs behind the scenes to compute the answer. It makes a derived value look like
> a plain attribute.

Let's run the real class against a stub model. First, a one-time setup cell so
`import src...` works no matter where Jupyter's working directory landed:


In [ ]:
# Setup: put the repo root on sys.path (walk up until we find src/).
import sys, pathlib

root = pathlib.Path.cwd()
while not (root / "src").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))
print("repo root:", root)


In [ ]:
import torch
from src.cache.capture import SampleCapture

class StubVibeVoice:
    # same signature as the real hook point; "latent" = first 4 dims, halved
    def sample_speech_tokens(self, condition, neg_condition=None, cfg_scale=1.3):
        return condition[..., :4] * 0.5

model = StubVibeVoice()
cap = SampleCapture(model)
with cap:
    for t in range(5):                                   # pretend: 5 speech frames
        model.sample_speech_tokens(torch.randn(1, 12))   # model runs as normal

print("frames captured:", cap.num_frames)                # @property, no parentheses

utt = cap.to_utterance("demo-000", "hello world")        # runs the alignment guard too
print("hidden:", tuple(utt.hidden.shape), utt.hidden.dtype)
print("latent:", tuple(utt.latent.shape), utt.latent.dtype)
print("method restored after 'with'?", "sample_speech_tokens" not in vars(model))


## 2. The alignment guard — and the April 7 story

Here is `src/cache/alignment.py`, complete. It is ten lines of code and it is load-bearing:

```python
def assert_frame_aligned(hidden: torch.Tensor, latent: torch.Tensor) -> None:
    if hidden.ndim != 3 or latent.ndim != 3:
        raise ValueError(f"expected [B, T, d] tensors, got {hidden.shape} and {latent.shape}")
    if hidden.shape[:2] != latent.shape[:2]:
        raise ValueError(
            f"frame misalignment: hidden {tuple(hidden.shape)} vs latent {tuple(latent.shape)} "
            "-- this is the April 7 failure signature, do not proceed"
        )
    if not torch.isfinite(hidden).all() or not torch.isfinite(latent).all():
        raise ValueError("non-finite values in cached pair -- refusing to write")
```

Three checks: both tensors are `[batch, time, features]`; both have the **same number of
frames** (`shape[:2]` = batch and time must match — feature widths may legitimately
differ, 1536 vs 64); and nothing is NaN or infinite.

### Why the error message names a date

On **April 7, 2026** (finding **N2** in `docs/negative-results.md`, from the predecessor
project TransplantTTS), a flow head with this exact architecture and recipe trained to a
healthy-looking loss — and produced audio that was *sharp but unintelligible*. The
numbers, from the record:

- loudness looked completely normal: RMS −16.0 dBFS vs −15.6 for the baseline;
- but the **zero-crossing rate was ~1,248 per second**, when real speech lives in the
  **3,000–8,000/s** band. (Zero-crossing rate = how often the waveform crosses zero;
  a crude proxy for high-frequency content — consonants push it way up. A "voice" with
  ZCR 1,250 is a low mumble with no consonants in it.)
- texture was actually *sharper* than the baseline — the head **was** generating —
  just not the right phoneme content. Sharpness without meaning.

The root cause there turned out to be the conditioning signal (hidden states from an
MSE-pretrained backbone — the whole reason LongFlow conditions on VibeVoice's own states
instead). But the episode taught us the *shape* of this failure class: **the pipeline
confidently produces garbage while every loss curve looks healthy**. A silently
misaligned cache — frame 12's thought paired with frame 13's sound — would produce
exactly that signature, and the pre-registered FAIL branch of the P1 gate even names
"capture correctness" as the most likely bug class. Hence the assert's wording.

### Why check at **every write and every read**

- **Write** (`SampleCapture.to_utterance`, `BatchedSampleCapture.split_utterances`):
  catch a capture bug at the moment it happens, with the utterance ID in hand.
- **Read** (`load_utterance` re-runs the guard on every file it loads): files get
  truncated, Google Drive syncs get interrupted, and code drifts between cache-time and
  train-time. From `capture.py`:

```python
def load_utterance(path) -> UtteranceCache:
    d = torch.load(path, weights_only=True)
    utt = UtteranceCache(**d)
    assert_frame_aligned(
        utt.hidden.float().unsqueeze(0), utt.latent.float().unsqueeze(0)
    )  # verify on read too — cheap insurance against corrupt/truncated files
    return utt
```

The check costs microseconds. A missed misalignment costs a GPU run **plus every
conclusion drawn from it**. Cheap insurance. Watch it refuse:


In [ ]:
import torch
from src.cache.alignment import assert_frame_aligned

hidden = torch.randn(1, 50, 12)   # 50 frames of "thought"
latent = torch.randn(1, 49, 4)    # 49 frames of "sound" -- off by one!
try:
    assert_frame_aligned(hidden, latent)
except ValueError as e:
    print("REFUSED:", e)

print()
assert_frame_aligned(torch.randn(1, 50, 12), torch.randn(1, 50, 4))
print("aligned pair (50 vs 50 frames, different widths): passes silently")

print()
bad = torch.randn(1, 50, 4)
bad[0, 3, 2] = float("nan")       # one poisoned number out of 200
try:
    assert_frame_aligned(torch.randn(1, 50, 12), bad)
except ValueError as e:
    print("REFUSED:", e)


## 3. Storage decisions: fp16, and why the checkpoint carries statistics

### fp16 halves the disk

Tensors default to `float32` — 4 bytes per number. The cache stores everything as
`float16` — 2 bytes — because training pairs don't need more precision than the model
computed them at (VibeVoice itself runs in bf16). That single decision halves the cache.
The arithmetic:


In [ ]:
d_model, d_latent = 1536, 64        # thought width + sound width
bytes_per_frame = (d_model + d_latent) * 2      # fp16 = 2 bytes per number
frames_per_utt = 41450 / 800        # measured: the gate run's 800 utts -> 41,450 pairs

def cache_size(n_utts):
    return n_utts * frames_per_utt * bytes_per_frame

print(f"per frame:    {bytes_per_frame:,} bytes  (~{bytes_per_frame/1024:.1f} KiB)")
print(f"800 utts:     {cache_size(800)/1e6:>7.0f} MB in fp16   ({2*cache_size(800)/1e6:.0f} MB if fp32)")
print(f"10,000 utts:  {cache_size(10_000)/1e9:>7.2f} GB in fp16")
print(f"75,000 utts:  {cache_size(75_000)/1e9:>7.1f} GB in fp16   ({2*cache_size(75_000)/1e9:.0f} GB if fp32)")


So the full 75K-utterance run is a ~12.5 GB cache instead of ~25 GB — the difference
between "fits on the free tier of Google Drive alongside everything else" and "doesn't".

### Loading the cache: `load_pairs`

Training doesn't care about utterance boundaries — the head is per-frame — so
`src/flow_head/trainer.py` flattens every cached utterance into one big pile of
(thought, sound) rows:

```python
def load_pairs(cache_dir, limit: int | None = None) -> PairData:
    """Flatten cached utterances into frame pairs; compute latent stats."""
    files = sorted(Path(cache_dir).glob("*.pt"))[:limit]
    ...
    mean = latent.mean(dim=0)
    std = latent.std(dim=0).clamp_min(1e-4)
    return PairData(hidden=hidden, latent=(latent - mean) / std, mean=mean, std=std)
```

> **Python syntax note — generators.** `Path(cache_dir).glob("*.pt")` doesn't return a
> list — it returns a **generator**, an object that produces matches one at a time, only
> when asked. Nothing is scanned until something consumes it; here `sorted(...)` consumes
> it all and returns a real (sorted) list. Generators let you iterate over huge sequences
> without ever holding them all in memory.

> **Python syntax note — `@property` again.** `PairData.d_model` is defined as a
> `@property` returning `self.hidden.shape[1]` — so the width is always *derived from the
> actual data*, never stored as a second copy that could drift out of sync.

### Why standardize the latents?

The last line above rescales every latent dimension to mean 0, standard deviation 1
(and `clamp_min(1e-4)` stops a nearly-constant dimension from causing a divide-by-zero).
Two reasons, both from how flow matching works:

1. **Flow matching starts from N(0, 1) noise.** The head learns to transport pure
   standard-normal noise into data. If the data is *also* roughly standard-normal per
   dimension, the journey is short and similar in every dimension — which matters a lot
   when you only take 4 solver steps.
2. **Matched scales across dimensions.** The loss is a plain MSE over all 64 latent
   dims. If one dim naturally ranged ±40 and another ±0.1, the big one would dominate the
   loss and the small one would never be learned properly. Standardizing puts every
   dimension on equal footing.

### Why mean/std must ship *inside* the checkpoint

The head now lives its whole life in standardized space — but the frozen σ-VAE decoder
expects latents in the **original** space. So sampling must undo the transform, and the
checkpoint format guarantees it always can:

```python
def save_checkpoint(path, head, ema, data, step):
    torch.save({
        "config": vars(head.cfg),
        "model": head.state_dict(),
        "ema": ema.shadow,
        "latent_mean": data.mean,     # <- the stats travel WITH the weights
        "latent_std": data.std,
        "step": step,
    }, path)
```

and `sample_latents` ends with the inverse transform: `return z * latent_std + latent_mean`.

If the stats lived in a separate file (or worse, were recomputed later from a *different*
cache), a checkpoint could silently generate latents in the wrong units — numerically
plausible, acoustically garbage. A checkpoint must be a complete, self-contained recipe
for producing decoder-ready latents.


In [ ]:
# Standardize -> invert round-trip: the transform must be exactly reversible.
import torch

torch.manual_seed(0)
latent = torch.randn(1000, 4) * torch.tensor([40.0, 3.0, 0.1, 1.0]) + torch.tensor([5.0, -2.0, 0.0, 0.3])

mean = latent.mean(dim=0)
std = latent.std(dim=0).clamp_min(1e-4)
z = (latent - mean) / std                     # what the head trains on

print("raw per-dim std:         ", [f"{s:.2f}" for s in latent.std(dim=0)])
print("standardized per-dim std:", [f"{s:.2f}" for s in z.std(dim=0)])

restored = z * std + mean                     # what sample_latents does on the way out
print("max round-trip error:    ", float((restored - latent).abs().max()))


## 4. The trainer: a borrowed recipe, a moving average, and one gotcha

The training loop in `src/flow_head/trainer.py` is deliberately boring. The optimizer
line is:

```python
opt = torch.optim.Adam(head.parameters(), lr=lr, betas=(0.9, 0.95), weight_decay=0.0)
```

Learning rate 2e-4, betas (0.9, 0.95), no weight decay, constant LR — none of this was
tuned by us. It's the published recipe from the MeanFlow-paper lineage, pinned in
`docs/resources.md` §2. (Adam's betas control how much memory the optimizer keeps of past
gradients; 0.95 for the second beta — instead of the 0.999 default — makes it adapt
faster, a common choice in modern generative-model recipes.) When a recipe is known to
work for exactly this kind of head, copying it removes a whole axis of things that can go
wrong. Two lines later:

```python
torch.nn.utils.clip_grad_norm_(head.parameters(), 1.0)
```

**Gradient clipping in one sentence:** if a freak batch produces a gradient whose overall
length exceeds 1.0, it gets rescaled down to length 1.0 — one bad batch can *nudge* the
weights but never *fling* them.

> **Python syntax note — f-strings with format specs.** The trainer logs with
> `print(f"step {step}/{steps}  loss {recent:.4f}")`. An `f"..."` string evaluates the
> expressions inside `{}` and splices them in; the part after the colon is a **format
> spec** — `:.4f` means "as a decimal with 4 digits after the point", `:,` adds thousands
> separators, `:>7.2f` right-aligns in 7 characters. (You've seen `!r` in error messages
> too: it formats the value as its `repr`, so strings keep their quotes.)

### EMA: the bathtub thermostat

The trainer keeps **two** copies of the head's weights:

```python
class EMA:
    def __init__(self, model, decay: float = 0.9999):
        self.decay = decay
        self.shadow = {n: p.detach().clone() for n, p in model.named_parameters()}

    @torch.no_grad()
    def update(self, model):
        for n, p in model.named_parameters():
            self.shadow[n].lerp_(p.detach(), 1.0 - self.decay)
```

The **live** weights jump around with every noisy mini-batch — like a thermostat's
reading flickering with every draft of air. The **shadow** copy is the bathtub of water:
after every step it moves a tiny fraction `(1 - decay)` of the way toward the live
weights (`lerp_` = linear interpolation, in place), so it tracks the *average recent*
position and ignores the flicker. At the end you evaluate and ship the calm bathtub, not
the jittery thermostat — for generative models this reliably samples better.

### The gotcha we caught: decay vs. run length

An EMA with decay `d` has a memory horizon of roughly `1/(1-d)` steps. Decay 0.9999 —
the literature default, meant for 100K+ step runs — remembers the last **10,000** steps.
Run it for only a few hundred steps and the shadow is still almost entirely... the random
initialization. After `n` updates, the fraction of the shadow that is *still the initial
weights* is exactly `decay ** n`:


In [ ]:
for decay, steps in [(0.9999, 300), (0.9999, 5000), (0.999, 5000)]:
    still_init = decay ** steps
    print(f"decay {decay}, {steps:>5} steps -> shadow is {still_init:.1%} initialization")


Decay 0.9999 over 300 steps: the "trained" EMA weights are **~97% random init**. Even over
the full 5,000-step gate run they'd still be 61% init — a checkpoint that looks trained
(the live weights are fine!) but whose EMA copy, the one `load_checkpoint(use_ema=True)`
hands out *by default*, is mostly noise. That's why `experiments/p1_flow_head/NOTES.md`
pins:

> Adam lr 2e-4, betas (0.9, 0.95), bs 1024, 5K steps, **EMA 0.999** (0.9999 would stay
> ~at init over 5K — see trainer test), fp32

Decay 0.999 has a ~1,000-step horizon: comfortably inside a 5K run. Watch the real `EMA`
class demonstrate both behaviours:


In [ ]:
import torch
from src.flow_head.trainer import EMA

def shadow_progress(decay, n_updates):
    net = torch.nn.Linear(8, 8)
    ema = EMA(net, decay=decay)                    # shadow starts AT the init weights
    with torch.no_grad():
        for p in net.parameters():
            p.add_(1.0)                            # pretend training moved every weight by +1
    for _ in range(n_updates):
        ema.update(net)                            # shadow chases the live weights
    # how far did the shadow travel toward the trained weights? (0 = stuck at init, 1 = arrived)
    name, live = next(iter(net.named_parameters()))
    init = live.detach() - 1.0
    return float((ema.shadow[name] - init).mean() / 1.0)

for decay, n in [(0.9999, 300), (0.9999, 5000), (0.999, 5000)]:
    print(f"decay {decay}, {n:>5} updates -> shadow has covered {shadow_progress(decay, n):.1%} of the distance")


## 5. The batched un-shuffling puzzle

Caching one utterance at a time worked but cost ~8.4 s each — 75K utterances would be
~187 GPU-hours. Generating a **batch** of 8 at once cut it to 1.7 s/utt (**4.9×**,
validated on the real model, `experiments/p1_flow_head/NOTES.md`). But batching creates a
genuinely tricky attribution puzzle:

- Batch elements are different sentences, so they **finish at different steps**.
- Some steps, an element emits a speech frame; other steps it emits text tokens or has
  already ended. So each `sample_speech_tokens` call carries condition/latent **rows only
  for the batch elements emitting a speech frame at that step** (VibeVoice slices the
  batch with `diffusion_indices`, in ascending batch order).
- The capture wrapper just sees a stream of anonymous row-blocks: 3 rows, then 2 rows,
  then 3, ... **Which row belongs to which utterance?**

`BatchedSampleCapture.split_utterances` answers by **replaying the token streams**. After
generation we know exactly which tokens each element produced at each step. So: walk the
steps; at step *t*, the elements whose token is the speech-frame id were the active ones;
capture call *k* must therefore correspond to the *k*-th step that had any active
elements, and its rows map to that step's active list, in order.

That "must therefore" is doing a lot of work — so it's not trusted, it's **checked**.
Three hard invariants, each a `RuntimeError` if violated:

1. **call/step count**: number of capture calls == number of steps with ≥1 active frame;
2. **row/active count**: each call's row count == that step's number of active elements;
3. **per-element total**: frames assigned to element *b* == frame tokens in *b*'s stream.

This is the **refuse-loudly philosophy**: a crash costs a re-run; a silent misattribution
trains the head on (speaker A's thought, speaker B's sound) and poisons every conclusion
downstream — the April 7 lesson again, in batch form.

### The real ~1% edge case

The 10K-run-scale work surfaced one legitimate violation, now handled in `capture.py`:

```python
# End-of-generation edge case (observed ~1% of batches, 2026-07-07 run):
# a frame token emitted at the very final step is never rendered —
# generation stops before its sample_speech_tokens call. Exactly one
# trailing frame-step with no call is therefore attributable and safe
# to drop; anything else stays a hard abort.
dropped_final = None
if len(frame_steps) == len(self.calls) + 1:
    dropped_final = frame_steps.pop()
```

If an element's *very last* token is a frame token, generation halts before that frame's
latent ever gets sampled — so there's exactly one more frame-step than capture calls.
That specific, provable case is tolerated (drop the orphan step, lower that element's
expected count by one); **two** missing calls is not attributable and still aborts.

The cell below is the same simulation the unit tests use
(`tests/test_batched_capture.py`): row values encode `(element*100 + frame_index)` so we
can verify attribution *exactly* by eye.


In [ ]:
import torch
from src.cache.capture import BatchedSampleCapture

DM, DL = 12, 4          # toy widths
FRAME, END = 100, 102   # toy token ids: 100 = "speech frame", 102 = "end"

class StubModel:
    def sample_speech_tokens(self, condition, neg_condition=None, cfg_scale=1.3):
        return condition[..., :DL] * 2.0

def simulate(streams):
    """Replay a batched 'generation': at each step, one call carrying rows
    for exactly the elements emitting a frame. Row values encode
    (element*100 + that element's frame index) so attribution is checkable."""
    model = StubModel()
    frame_counter = [0] * len(streams)
    cap = BatchedSampleCapture(model)
    with cap:
        for t in range(max(len(s) for s in streams)):
            active = [b for b, s in enumerate(streams) if t < len(s) and s[t] == FRAME]
            if not active:
                continue
            rows = []
            for b in active:
                rows.append(torch.full((DM,), float(b * 100 + frame_counter[b])))
                frame_counter[b] += 1
            model.sample_speech_tokens(torch.stack(rows), None, 1.3)
    return cap

streams = [
    [FRAME, FRAME, FRAME, END],                # element 0: 3 frames, finishes early
    [FRAME, FRAME, FRAME, FRAME, FRAME, END],  # element 1: 5 frames
    [FRAME, END, FRAME, FRAME, END],           # element 2: a gap mid-stream
]
cap = simulate(streams)
print("raw capture: rows per call =", [c.shape[0] for c, _ in cap.calls], "(anonymous!)")

parts = cap.split_utterances(streams, frame_id=FRAME)
for b, (hidden, latent) in enumerate(parts):
    ids = [int(hidden[i, 0]) for i in range(hidden.shape[0])]
    print(f"element {b}: {hidden.shape[0]} frames, row ids {ids}  <- b*100+frame, in order")


In [ ]:
# Refuse-loudly: sabotage a call, watch the invariants catch it.
cap = simulate(streams)
cond, lat = cap.calls[0]
cap.calls[0] = (cond[:1], lat[:1])          # drop a row from the first call
try:
    cap.split_utterances(streams, frame_id=FRAME)
except RuntimeError as e:
    print("REFUSED:", e)

print()
# The ~1-percent edge case: element 1 ends ON a frame token; that final frame's
# sample_speech_tokens call never happens. One orphan step -> tolerated.
streams2 = [[FRAME, FRAME, END], [FRAME, FRAME, FRAME]]
cap2 = simulate(streams2)
cap2.calls.pop()                            # generation stopped before the last call
parts2 = cap2.split_utterances(streams2, frame_id=FRAME)
print("edge case handled: frames per element =", [p[0].shape[0] for p in parts2],
      " (element 1 had 3 frame tokens, last one unrendered -> 2)")

# ...but TWO missing calls is not attributable: hard abort.
cap3 = simulate(streams2)
cap3.calls.pop(); cap3.calls.pop()
try:
    cap3.split_utterances(streams2, frame_id=FRAME)
except RuntimeError as e:
    print("REFUSED:", e)


## 6. How we measure quality

`src/eval/metrics.py` computes three families of numbers per clip. All the heavy models
are lazy-loaded and cached at module level, so a sweep over 50 clips pays the load cost
once. (Don't run that module here — it downloads Whisper and ECAPA weights on first use;
the cells below are dependency-free toys of the same math.)

### WER — "did it say the right words?"

```python
def wer(path, reference_text: str) -> dict:
    segments, _info = _whisper().transcribe(str(path), language="en", beam_size=5)
    hyp = " ".join(s.text for s in segments)
    norm = _normalizer()
    ref_n, hyp_n = norm(reference_text), norm(hyp)
    return {"wer": jiwer.wer(ref_n, hyp_n), "transcript": hyp.strip()}
```

Whisper (large-v3, int8 on CPU) *listens* to the clip and writes down what it heard;
`jiwer` then counts the minimum number of word-level edits (substitutions, insertions,
deletions) needed to turn the reference into the transcript, divided by the reference
length. Both texts pass through `whisper-normalizer` first so "Dr." vs "doctor" or
"2" vs "two" don't count as errors. WER 0.0 = perfect; 0.03 means ~1 word in 33 off.

Here's the edit-distance intuition without jiwer — the classic dynamic-programming grid:


In [ ]:
def word_edits(ref, hyp):
    """Minimum substitutions+insertions+deletions to turn ref into hyp (word level)."""
    r, h = ref.split(), hyp.split()
    d = [[0] * (len(h) + 1) for _ in range(len(r) + 1)]
    for i in range(len(r) + 1):
        d[i][0] = i                     # delete everything
    for j in range(len(h) + 1):
        d[0][j] = j                     # insert everything
    for i in range(1, len(r) + 1):
        for j in range(1, len(h) + 1):
            d[i][j] = min(d[i - 1][j] + 1,                          # deletion
                          d[i][j - 1] + 1,                          # insertion
                          d[i - 1][j - 1] + (r[i - 1] != h[j - 1])) # substitution (free if equal)
    return d[-1][-1]

ref = "the quick brown fox jumps over the lazy dog"
for hyp in [ref,
            "the quick brown fox jumped over the lazy dog",
            "quick brown fox jumps over a lazy dog dog"]:
    e = word_edits(ref, hyp)
    print(f"edits={e}  WER={e / len(ref.split()):.3f}   '{hyp}'")


### Speaker similarity — "does it still sound like the same person?"

```python
def speaker_similarity(path, reference_path) -> float:
    model = _ecapa()
    embs = []
    for p in (path, reference_path):
        wav = load_16k_mono(p).unsqueeze(0)
        with torch.no_grad():
            embs.append(model.encode_batch(wav).squeeze())
    return float(torch.nn.functional.cosine_similarity(embs[0], embs[1], dim=-1))
```

ECAPA-TDNN squeezes an entire clip into one **embedding** — a ~192-dimensional vector
positioned so that clips of the *same voice* point in the *same direction*, regardless of
what words are spoken. **Cosine similarity** then measures the angle between two such
vectors: +1 = same direction, 0 = unrelated, −1 = opposite. Crucially it ignores vector
*length* — a loud clip and a quiet clip of the same speaker still score ~1. Same math
in 2D, where you can picture the arrows:


In [ ]:
import math

def cosine(a, b):
    dot = a[0] * b[0] + a[1] * b[1]
    return dot / (math.hypot(*a) * math.hypot(*b))

pairs = [((1, 0), (1, 0),   "identical direction"),
         ((1, 0), (5, 0),   "same direction, 5x longer (louder) -- length ignored"),
         ((1, 0), (1, 1),   "45 degrees apart"),
         ((1, 0), (0, 1),   "perpendicular (unrelated)"),
         ((1, 0), (-1, 0),  "opposite")]
for a, b, label in pairs:
    print(f"{str(a):>8} vs {str(b):>8}: cosine {cosine(a, b):+.3f}   {label}")


### Prosody — in two sentences

`prosody()` uses parselmouth (a Python wrapper around the Praat phonetics toolkit) to
extract **F0** — the pitch contour — reporting its mean and spread over voiced frames,
plus overall RMS loudness in dBFS. It's the "does the melody and energy of the speech
look normal?" check that catches monotone, warble, or the wrong register even when the
words are right.

### The gate-check method: decide the verdict *before* the experiment

The project's discipline, visible in `experiments/p1_flow_head/NOTES.md`: PASS / PARTIAL
/ FAIL criteria are **pre-registered** — written down *before* the run, so the results
can't sweet-talk us into moving the goalposts. The P1 gate table, verbatim:

> | Verdict | Condition |
> |---|---|
> | PASS | (a) roundtrip sanity: cached ground-truth latents decode to normal VibeVoice-quality speech; (b) training loss decreases smoothly, no NaN; (c) flow-head samples at 4 NFE decode to intelligible speech, recognizably the prompt speaker, ZCR in the 3–8k/s band |
> | PARTIAL | Intelligible but clearly degraded vs roundtrip → try NFE 8/16, more steps, or width 768 before re-gating |
> | FAIL | April-7 signature (sharp but unintelligible, ZCR ≪ 3k) at any NFE despite healthy loss → stop, re-examine capture correctness first |
>
> Listening step: roundtrip clip AND flow-decoded clips, A/B against the original
> generation. **Never skipped.**

That last line is April 7's most durable lesson: on April 7 the summary metrics looked
fine — loss healthy, loudness normal, texture "sharp" — and only *listening* revealed
the audio was unintelligible (the ZCR number then confirmed what ears already knew).
Ears also work in the other direction: in the July gate run, listening correctly
attributed a small onset artifact to VibeVoice *itself* (present identically in the
teacher clips) rather than to our head — saving a pointless "fix". The gate verdict,
for the record: **PASS** — flow head at 4 NFE, WER 0.030 vs teacher, speaker similarity
0.984.


## 7. Go deeper

- **Whisper** (the transcriber behind our WER): *Robust Speech Recognition via Large-Scale
  Weak Supervision* — [arXiv:2212.04356](https://arxiv.org/abs/2212.04356)
- **ECAPA-TDNN** (the speaker-embedding model): *ECAPA-TDNN: Emphasized Channel Attention,
  Propagation and Aggregation in TDNN Based Speaker Verification* —
  [arXiv:2005.07143](https://arxiv.org/abs/2005.07143)
- **SpeechBrain** (the toolkit that serves us ECAPA): [speechbrain.github.io](https://speechbrain.github.io/)
- **jiwer** (the WER calculator): [github.com/jitsi/jiwer](https://github.com/jitsi/jiwer)
- **Karpathy, "A Recipe for Training Neural Networks"** — the mindset behind our
  gate-check discipline (fix seeds, look at the data, get the dumb baseline right first):
  [karpathy.github.io/2019/04/25/recipe](https://karpathy.github.io/2019/04/25/recipe/)
- **`docs/resources.md`** — this repo's own pin registry: every model, library, dataset,
  and recipe version we depend on, with verification dates. When something in the code
  says "per resources.md §2", that's where the receipt lives.
